# QuantJourney SDK - Institutional Crowding + Liquidity Capacity + Historical Stress

This notebook demonstrates a QuantJourney SDK workflow that combines ownership, ADV, short-interest and stress context into a crowding and capacity screen.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

In [ ]:
import os
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2018-01-01')
END = os.getenv('QJ_EXAMPLE_END', '2026-06-06')
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 4.5), 'axes.grid': True, 'grid.alpha': 0.25})

def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    v = unwrap(payload)
    if v is None:
        return []
    if isinstance(v, list):
        return v
    if isinstance(v, dict):
        for k in ('rows', 'data', 'items', 'holdings'):
            if isinstance(v.get(k), list):
                return v[k]
        return [v]
    return []

def price_panel(symbols, start=START, end=END):
    prices, vols = ({}, {})
    for s in symbols:
        p = qj.eod.get_historical_prices(symbol=s, start_date=start, end_date=end)
        rows = as_rows(p)
        df = pd.DataFrame(rows)
        if not df.empty:
            df['date'] = pd.to_datetime(df.get('date'))
            prices[s] = pd.to_numeric(df.get('adjusted_close').fillna(df.get('close')), errors='coerce')
            vols[s] = pd.to_numeric(df.get('volume'), errors='coerce')
    px = pd.DataFrame(prices).dropna(how='all')
    vv = pd.DataFrame(vols).reindex(px.index)
    return (px, vv)

def get_13f_concentration(symbols):
    out = []
    for s in symbols:
        h = as_rows(qj.fmp.get_institutional_holders(symbol=s))
        vals = [pd.to_numeric(x.get('value') or x.get('marketValue') or x.get('shares'), errors='coerce') for x in h]
        vals = pd.Series(vals).dropna()
        conc = vals.nlargest(10).sum() / vals.sum() if vals.sum() > 0 else np.nan
        out.append({'symbol': s, 'holder_count': len(h), 'top10_conc': conc})
    return pd.DataFrame(out).set_index('symbol')

def get_short_interest(symbols):
    res = {}
    for s in symbols:
        si = qj.finra.get_short_interest(symbol=s)
        res[s] = unwrap(si) or {}
    return res


In [ ]:
symbols = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META']
px, vol = price_panel(symbols)
adv = (px * vol).rolling(63).mean().iloc[-1]
crowd = get_13f_concentration(symbols)
crowd['adv_usd'] = adv.reindex(crowd.index)
crowd['est_days_1pct'] = crowd.get('top10_conc', 0.3) * 10000000000.0 / (crowd['adv_usd'].replace(0, np.nan) * 0.01)
print('Crowding + liquidity proxy:\n', crowd.round(2))
vix = qj.cboe.get_vix_data()
if isinstance(vix, dict):
    vix = pd.DataFrame(as_rows(vix))
if not vix.empty and 'date' in vix:
    vix = vix.set_index(pd.to_datetime(vix['date']))['close'].sort_index()
    high_stress = vix > vix.quantile(0.85)
    print('\nHigh stress days sample:', high_stress.sum())
crowd[['top10_conc', 'adv_usd']].plot(kind='bar', subplots=True, title='Crowding vs Liquidity Capacity')
plt.tight_layout()
plt.show()
print('This example shows how 13F concentration + real ADV + historical vol stress can be fused without any internal PMS data.')


## Notes

Multi-source example: regulatory ownership (13F) + pricing/volume + microstructure (shorts) + vol/macro stress.
All numbers are illustrative. Real capacity models need better participation assumptions and full 13F history.
No live portfolio or IBOR data used.